In [1]:
# Add the 'src' directory to the sys.path
import sys
sys.path.append('../../src')

In [2]:
import os
import time
import ssl
from elasticsearch import Elasticsearch
from etl import DataReader, DataLoader

In [3]:
current_dir = os.getcwd()
normalized_data = current_dir + "/" + "judiciary_data.jsonl"
CA_CERT_PATH = "/home/felipe/dev/projects/poc-elasticsearch/containers/esconfig/certs/http_ca.crt"
JUDICIARY_SOURCE_DATA = "/home/felipe/dev/projects/judiciary-system/notebooks/data/output.jsonl"
ES_JUDICIARY_INDEX_NAME = "judiciary"

In [4]:
ssl_context = ssl.create_default_context(cafile=CA_CERT_PATH)

es = Elasticsearch(
    ["https://localhost:9200"],
    basic_auth=("elastic", "worksfine"),
    ssl_context=ssl_context,
    request_timeout=60,
)

es

<Elasticsearch(['https://localhost:9200'])>

In [5]:
es.cluster.health()

ObjectApiResponse({'cluster_name': 'docker-cluster', 'status': 'yellow', 'timed_out': False, 'number_of_nodes': 1, 'number_of_data_nodes': 1, 'active_primary_shards': 1, 'active_shards': 1, 'relocating_shards': 0, 'initializing_shards': 0, 'unassigned_shards': 1, 'delayed_unassigned_shards': 0, 'number_of_pending_tasks': 0, 'number_of_in_flight_fetch': 0, 'task_max_waiting_in_queue_millis': 0, 'active_shards_percent_as_number': 50.0})

In [6]:
reader = DataReader(file_path=normalized_data)
loader = DataLoader(reader)

In [7]:
# delete index if exists
es.options(ignore_status=[400,404]).indices.delete(index=ES_JUDICIARY_INDEX_NAME)

ObjectApiResponse({'acknowledged': True})

In [8]:
%%time
# index whole dataset into elasticsearch
loader.load_into_es(es_client=es, index_name=ES_JUDICIARY_INDEX_NAME)

CPU times: user 30.7 s, sys: 2.08 s, total: 32.8 s
Wall time: 4min 46s


In [9]:
def get_total_documents_in_index(es: Elasticsearch, index_name: str) -> int:
    """Returns total number of documents in es index"""

    response = es.count(
        index=index_name,
        body={
          "query": {
            "match_all": {},
          },
        },
    )

    return response["count"]

In [10]:
time.sleep(10)

In [11]:
es_query = {
    "query": {
        "bool": {
            "must": [
                {
                    "multi_match": {
                        "query": "brigandage",
                        "fields": ["document_text"],
                        "slop": 999,
                    }
                },
                # {
                #     "match": {
                #         "document_text": "brigandage"
                #     }
                # },
            ],
        }
    }
}


print("Total documents indexed:", get_total_documents_in_index(es=es, index_name=ES_JUDICIARY_INDEX_NAME))

response = es.search(
    index=ES_JUDICIARY_INDEX_NAME,
    body=es_query,
)

es_and_total = response["hits"]["total"]["value"]
print('query:brigandage Results --->', es_and_total)

Total documents indexed: 22296
query:brigandage Results ---> 205


In [12]:
es_query = {
    "query": {
        "bool": {
            "must": [
                {
                    "multi_match": {
                        "query": "divorce",
                        "fields": ["document_text"],
                        "slop": 999,
                    }
                },
                # {
                #     "match": {
                #         "document_text": "brigandage"
                #     }
                # },
            ],
        }
    }
}


print("Total documents indexed:", get_total_documents_in_index(es=es, index_name=ES_JUDICIARY_INDEX_NAME))

response = es.search(
    index=ES_JUDICIARY_INDEX_NAME,
    body=es_query,
)

es_and_total = response["hits"]["total"]["value"]
print('query:divorce Results --->', es_and_total)

Total documents indexed: 22296
query:divorce Results ---> 928
